In [13]:

from nhs_waiting_lists.constants import PROVIDER_CODES
from nhs_waiting_lists.core.waiting_list_db import WaitingListsDB



In [15]:

from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text

from nhs_waiting_lists import (
    __app_name__,
)
from nhs_waiting_lists.constants import proj_db_path
from nhs_waiting_lists.utils.xdg import XDGBasedir

project_root = Path(XDGBasedir.get_data_dir(__app_name__))

DB_PATH = project_root / proj_db_path / "nhs_provider_data2.db"
DATA_DIR = project_root / "data"
engine = create_engine(f"sqlite:///{DB_PATH}", echo=True)
# Base.metadata.create_all(engine)

/home/tomhodder/Sync/projects/data/nhs_england_data/src/nhs_waiting_lists/models/t_v_consolidated.py:6: SAWarning: This declarative base already contains a class with the same class name and module name as nhs_waiting_lists.models.t_v_consolidated.VConsolidated, and will be replaced in the string-lookup table.
  class VConsolidated(Base):


In [18]:
db = WaitingListsDB()
df = db.rtt_by_provider("RAJ")
df


,provider,nhs_year,sum_untreated,sum_dna,unexplained_untreated


In [22]:


query = text("""
             SELECT c.provider,
                    c.nhs_year,
                    c.provider_name,
                    SUM(c.untreated) as sum_untreated

             FROM v_consolidated c

             WHERE c.treatment = 'C_999'
               AND c.provider IN ('RAJ')
             GROUP BY c.provider, c.nhs_year
             ORDER BY nhs_year ASC; \
             """).bindparams(
    # bindparam('provider_codes', expanding=True),
    # bindparam('treatment_codes', expanding=True)
)

df = pd.read_sql(query, conn, params={  # type: ignore[arg-type]
    'provider_codes': PROVIDER_CODES,
}
                 )
# df.query("provider == 'RAJ' and treatment == 'C_999' and period == '2025-08'")
df

,provider,nhs_year,provider_name,sum_untreated
0,RAJ,2021-22,Mid and South Essex NHS Foundation Trust,-75545
1,RAJ,2022-23,Mid and South Essex NHS Foundation Trust,-198779
2,RAJ,2023-24,Mid and South Essex NHS Foundation Trust,-230417
3,RAJ,2024-25,Mid and South Essex NHS Foundation Trust,-191949
4,RAJ,2025-26,Mid and South Essex NHS Foundation Trust,-39779


In [18]:


query = text("""
             SELECT reporting_period,
                    geography_level,
                    organisation_code,
                    measure_type,
                    measure,
                    SUM(measure_value) as dna_provider_sum

             FROM outpatients_activity oa
             WHERE oa.organisation_code == 'RAJ'
               AND oa.measure_type == 'Attendance Type'
               AND oa.measure LIKE 'Did not%'
               AND oa.geography_level == 'Provider'
             GROUP BY oa.organisation_code, oa.reporting_period, oa.measure_type;
             """).bindparams(
    # bindparam('provider_codes', expanding=True),
    # bindparam('treatment_codes', expanding=True)
)

df = pd.read_sql(query, conn, params={  # type: ignore[arg-type]
    'provider_codes': PROVIDER_CODES,
}
                 )
# df.query("provider == 'RAJ' and treatment == 'C_999' and period == '2025-08'")
df

,reporting_period,geography_level,organisation_code,measure_type,measure,dna_provider_sum
0,2017-18,Provider,RAJ,Attendance Type,Did not attend first appointment,32840.0
1,2018-19,Provider,RAJ,Attendance Type,Did not attend first appointment,30970.0
2,2019-20,Provider,RAJ,Attendance Type,Did not attend first appointment,31270.0
3,2020-21,Provider,RAJ,Attendance Type,Did not attend first appointment,67900.0
4,2021-22,Provider,RAJ,Attendance Type,Did not attend first appointment,98325.0
5,2022-23,Provider,RAJ,Attendance Type,Did not attend first appointment,108995.0
6,2023-24,Provider,RAJ,Attendance Type,Did not attend first appointment,112315.0
7,2024-25,Provider,RAJ,Attendance Type,Did not attend first appointment,109210.0


In [16]:

query = text("""
             SELECT organisation_code,
                    reporting_period,
                    measure_type,

                    -- sum of all rows in the group
                    SUM(measure_value)                                                   AS total_measure_value,

                    SUM(CASE WHEN measure LIKE 'Did not%' THEN measure_value ELSE 0 END) AS sum_dna,

                    -- conditional sub-totals
                    SUM(CASE
                            WHEN measure IN (
                                             'Did not attend first appointment',
                                             'Did not attend first tele consultation',
                                             'Did not attend subsequent appointment',
                                             'Did not attend subsequent tele consultation',
                                             'Did not attend, first/ subsequent/ tele unknown'
                                )
                                THEN measure_value
                            ELSE 0
                        END)                                                             AS total_did_not_attend,
                    SUM(CASE
                            WHEN measure IN (
                                             'Did not attend first appointment',
                                             'Did not attend first tele consultation',
                                             'Did not attend subsequent appointment',
                                             'Did not attend subsequent tele consultation',
                                             'Did not attend, first/ subsequent/ tele unknown'
                                )
                                THEN measure_value
                            ELSE 0
                        END) / SUM(measure_value)                                        AS pct_did_not_attend


             FROM outpatients_activity AS oa
             WHERE oa.measure_type = 'Attendance Type'
               AND oa.geography_level = 'National'
             GROUP BY oa.organisation_code,
                      oa.reporting_period,
                      oa.measure_type;

             """).bindparams(
    # bindparam('provider_codes', expanding=True),
    # bindparam('treatment_codes', expanding=True)
)

df = pd.read_sql(query, conn, params={  # type: ignore[arg-type]
    'provider_codes': PROVIDER_CODES,
}
                 )
# df.query("provider == 'RAJ' and treatment == 'C_999' and period == '2025-08'")
df

,organisation_code,reporting_period,measure_type,total_measure_value,sum_dna,total_did_not_attend,pct_did_not_attend
0,All,2017-18,Attendance Type,119378895.0,7984183.0,7984183.0,0.066881
1,All,2018-19,Attendance Type,123351435.0,7919660.0,7919660.0,0.064204
2,All,2019-20,Attendance Type,124927782.0,7695040.0,7695040.0,0.061596
3,All,2020-21,Attendance Type,101898658.0,5640749.0,5640749.0,0.055356
4,All,2021-22,Attendance Type,122325785.0,7826921.0,7826921.0,0.063984
5,ENG,2022-23,Attendance Type,124461569.0,8003452.0,8003452.0,0.064305
6,ENG,2023-24,Attendance Type,135445596.0,8020272.0,8020272.0,0.059214
7,ENG,2024-25,Attendance Type,146050990.0,8144111.0,8144111.0,0.055762
